# 02-1. Подготовка данных до запуска моделей тональности

В этом ноутбуке собраны шаги, которые нужны до модельного этапа: проверка входных файлов, нормализация новостей, аудит пропусков, построение доходностей и целевых переменных

## Блок 1. Импорт библиотек и настройка таблиц

In [1]:
import os
import warnings
from typing import List, Optional

import numpy as np
import pandas as pd
from tqdm import tqdm

from IPython.display import display

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 180)
pd.set_option("display.width", 200)

## Блок 2. Конфигурация проекта

Задаём пути, даты и базовые параметры подготовки данных

In [2]:
START_DATE = pd.Timestamp("2019-01-01")
END_DATE = pd.Timestamp("2023-12-31")

OUTPUT_DIR = "outputs_01"

NEWS_SOURCES = {
    "nasdaq": os.path.join(OUTPUT_DIR, "news_deduped_2019_2023.csv"),
}

SELECTED_EQUITIES_PATH = os.path.join(OUTPUT_DIR, "selected_equities_only_2019_2023.csv")
PRICES_DIR = "full_history/full_history"

CHUNK_SIZE = 200000
MAX_NEWS_PER_DAY = None

os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Папка вывода:", OUTPUT_DIR)

Папка вывода: outputs_01


## Блок 3. Проверка входных файлов

Проверяем, что входные файлы доступны, и загружаем список компаний

In [3]:
if not os.path.exists(SELECTED_EQUITIES_PATH):
    raise FileNotFoundError(f"Файл со списком тикеров не найден: {SELECTED_EQUITIES_PATH}")

missing_news = [fp for fp in NEWS_SOURCES.values() if not os.path.exists(fp)]
if missing_news:
    raise FileNotFoundError(f"Файл(ы) с новостями не найден(ы): {missing_news}")

if not os.path.isdir(PRICES_DIR):
    raise FileNotFoundError(f"Папка с ценами не найдена: {PRICES_DIR}")

companies = pd.read_csv(SELECTED_EQUITIES_PATH, low_memory=False)
companies["ticker"] = companies["ticker"].astype(str).str.strip()
TICKER_SET = set(companies["ticker"].tolist())

print("Тикеров в выборке:", len(TICKER_SET))
display(companies.head(5))

Тикеров в выборке: 2054


,ticker,company_name,sector,industry,exchange,quoteType,price_rows,price_min_date,price_max_date,coverage_pct,starts_in_2019,ends_in_2023,full_coverage,news_count,news_min_date,news_max_date
0,TSLA,"Tesla, Inc.",Consumer Cyclical,Auto Manufacturers,NMS,EQUITY,1257,2019-01-02,2023-12-28,100.0,True,True,True,10587,2019-07-01,2023-12-16
1,GOOG,Alphabet Inc.,Communication Services,Internet Content & Information,NMS,EQUITY,1257,2019-01-02,2023-12-28,100.0,True,True,True,9883,2019-01-02,2023-12-16
2,DIS,The Walt Disney Company,Communication Services,Entertainment,NYQ,EQUITY,1257,2019-01-02,2023-12-28,100.0,True,True,True,9654,2019-06-13,2023-12-16
3,BHFAL,"Brighthouse Financial, Inc.",NaN,NaN,NMS,EQUITY,1257,2019-01-02,2023-12-28,100.0,True,True,True,9614,2023-11-30,2023-12-16
4,AAPL,Apple Inc.,Technology,Consumer Electronics,NMS,EQUITY,1257,2019-01-02,2023-12-28,100.0,True,True,True,9338,2020-03-09,2023-12-16


## Блок 4. Нормализация новостей

Определяем функции для очистки и нормализации новостных данных

In [4]:
NEWS_COLS_RAW = [
    "Date", "Article_title", "Stock_symbol", "Url", "Publisher", "Author",
    "Article", "Lsa_summary", "Luhn_summary", "Textrank_summary", "Lexrank_summary",
]

def print_step(title: str):
    line = "=" * 88
    print("\n" + line)
    print(title)
    print(line)

def _clean_str(s: pd.Series) -> pd.Series:
    return s.fillna("").astype(str).str.strip()

def _normalize_news_cols(df: pd.DataFrame) -> pd.DataFrame:
    ren = {}
    if "Date" in df.columns and "date" not in df.columns:
        ren["Date"] = "date"
    if "Stock_symbol" in df.columns and "ticker" not in df.columns:
        ren["Stock_symbol"] = "ticker"
    if "Url" in df.columns and "url" not in df.columns:
        ren["Url"] = "url"
    if "Article_title" in df.columns and "title" not in df.columns:
        ren["Article_title"] = "title"
    if "Article" in df.columns and "article" not in df.columns:
        ren["Article"] = "article"
    if "Lsa_summary" in df.columns and "lsa_summary" not in df.columns:
        ren["Lsa_summary"] = "lsa_summary"
    if "Luhn_summary" in df.columns and "luhn_summary" not in df.columns:
        ren["Luhn_summary"] = "luhn_summary"
    if "Textrank_summary" in df.columns and "textrank_summary" not in df.columns:
        ren["Textrank_summary"] = "textrank_summary"
    if "Lexrank_summary" in df.columns and "lexrank_summary" not in df.columns:
        ren["Lexrank_summary"] = "lexrank_summary"
    if "Publisher" in df.columns and "publisher" not in df.columns:
        ren["Publisher"] = "publisher"
    if "Author" in df.columns and "author" not in df.columns:
        ren["Author"] = "author"
    return df.rename(columns=ren) if ren else df

def _ensure_cols(df: pd.DataFrame, cols: List[str]) -> pd.DataFrame:
    df = df.copy()
    for c in cols:
        if c not in df.columns:
            df[c] = np.nan
    return df

def _prep_news_chunk_base(raw: pd.DataFrame, source_tag: str) -> pd.DataFrame:
    ch = _normalize_news_cols(raw)

    need = [
        "date", "ticker", "url", "title", "article",
        "lsa_summary", "luhn_summary", "textrank_summary", "lexrank_summary",
        "publisher", "author"
    ]
    ch = _ensure_cols(ch, need)

    ch["ticker"] = _clean_str(ch["ticker"])
    ch["url"] = _clean_str(ch["url"])

    ch = ch[ch["ticker"] != ""]
    ch = ch[ch["url"] != ""]
    if ch.empty:
        return ch.iloc[0:0].copy()

    ch["date"] = pd.to_datetime(ch["date"], errors="coerce", utc=True).dt.tz_convert(None)
    ch = ch.dropna(subset=["date"])
    if ch.empty:
        return ch.iloc[0:0].copy()

    ch["day"] = ch["date"].dt.floor("D")
    ch = ch[(ch["day"] >= START_DATE) & (ch["day"] <= END_DATE)]
    if ch.empty:
        return ch.iloc[0:0].copy()

    ch = ch[ch["ticker"].isin(TICKER_SET)]
    if ch.empty:
        return ch.iloc[0:0].copy()

    if MAX_NEWS_PER_DAY is not None and int(MAX_NEWS_PER_DAY) > 0:
        ch = (
            ch.sort_values(["ticker", "day"], kind="mergesort")
              .groupby(["ticker", "day"], as_index=False, sort=False)
              .head(int(MAX_NEWS_PER_DAY))
        )

    ch = ch.drop_duplicates(subset=["ticker", "day", "url"])
    ch["source"] = source_tag

    keep_cols = [
        "ticker", "day", "url",
        "title", "article",
        "lsa_summary", "luhn_summary", "textrank_summary", "lexrank_summary",
        "publisher", "author",
        "source"
    ]
    return ch[keep_cols].reset_index(drop=True)

## Блок 5. Аудит пропусков

Проверяем пропуски в новостном файле после базовых фильтров

In [6]:
AUDIT_COLS = [
    "Date", "Stock_symbol", "Article_title", "Url", "Publisher", "Author",
    "Lsa_summary", "Luhn_summary", "Textrank_summary", "Lexrank_summary"
]

def audit_nan_counts(file_path: str, source_tag: str, max_chunks: Optional[int] = None) -> pd.DataFrame:
    header = pd.read_csv(file_path, nrows=0, low_memory=False)
    usecols = [c for c in AUDIT_COLS if c in header.columns]
    missing = [c for c in AUDIT_COLS if c not in header.columns]

    print(f"ИСТОЧНИК: {source_tag} | ФАЙЛ: {file_path}")
    print("Размер (МБ):", round(os.path.getsize(file_path)/1024/1024, 1))
    print("Используемые колонки:", usecols)
    if missing:
        print("Отсутствующие колонки в файле:", missing)

    total = 0
    nan_total = pd.Series(0, index=usecols, dtype="int64")

    total_f = 0
    nan_f = pd.Series(0, index=usecols, dtype="int64")

    examples_full = []
    examples_full_needed = 5

    reader = pd.read_csv(file_path, usecols=usecols, chunksize=CHUNK_SIZE, low_memory=False)

    for i, raw in enumerate(tqdm(reader, desc=f"audit {source_tag}")):
        if (max_chunks is not None) and (i >= int(max_chunks)):
            break

        total += len(raw)
        nan_total += raw[usecols].isna().sum().astype("int64")

        ch = _normalize_news_cols(raw)
        ch = _ensure_cols(ch, ["date","ticker","url","title","publisher","author",
                               "lsa_summary","luhn_summary","textrank_summary","lexrank_summary"])

        ch["ticker"] = _clean_str(ch["ticker"])
        ch["url"] = _clean_str(ch["url"])
        ch = ch[ch["ticker"] != ""]
        if ch.empty:
            continue

        ch["date"] = pd.to_datetime(ch["date"], errors="coerce", utc=True).dt.tz_localize(None)
        ch = ch.dropna(subset=["date"])
        if ch.empty:
            continue
        ch["day"] = ch["date"].dt.floor("D")
        ch = ch[(ch["day"] >= START_DATE) & (ch["day"] <= END_DATE)]
        ch = ch[ch["ticker"].isin(TICKER_SET)]
        ch = ch[ch["url"] != ""]
        if ch.empty:
            continue

        total_f += len(ch)

        ch_view = ch.copy()
        remap = {
            "date": "Date", "ticker": "Stock_symbol", "url": "Url", "title": "Article_title",
            "publisher": "Publisher", "author": "Author",
            "lsa_summary": "Lsa_summary", "luhn_summary": "Luhn_summary",
            "textrank_summary": "Textrank_summary", "lexrank_summary": "Lexrank_summary",
        }
        inv = {v:k for k,v in remap.items()}

        for c in usecols:
            nc = inv.get(c)
            if (nc is not None) and (nc in ch_view.columns):
                nan_f[c] += ch_view[nc].isna().sum()
            else:
                nan_f[c] += len(ch_view)

        if len(examples_full) < examples_full_needed:
            tmp = ch_view.copy()
            ex = pd.DataFrame(index=tmp.index)
            for c in usecols:
                nc = inv.get(c)
                if (nc is not None) and (nc in tmp.columns):
                    ex[c] = tmp[nc]
                else:
                    ex[c] = np.nan

            m = pd.Series(True, index=ex.index)
            for c in usecols:
                m &= ex[c].notna() & (_clean_str(ex[c]) != "")
            full_rows = ex.loc[m].head(examples_full_needed - len(examples_full))
            if not full_rows.empty:
                examples_full.extend(full_rows.to_dict(orient="records"))

    out = pd.DataFrame({"column": usecols})
    out["nan_total"] = out["column"].map(nan_total).astype("int64")
    out["nan_total_pct"] = (out["nan_total"] / max(total,1) * 100).round(2)

    out["nan_after_filters"] = out["column"].map(nan_f).astype("int64")
    out["nan_after_filters_pct"] = (out["nan_after_filters"] / max(total_f,1) * 100).round(2)

    print("Всего строк:", f"{total:,}")
    print("Строк после фильтров:", f"{total_f:,}", f"({(total_f/max(total,1)*100):.2f}%)")

    print("\nСводка по пропускам:")
    display(out.sort_values("nan_after_filters_pct", ascending=False))

    return out

audit_tables = {}
for tag, fp in NEWS_SOURCES.items():
    audit_tables[tag] = audit_nan_counts(fp, tag, max_chunks=None)

ИСТОЧНИК: nasdaq | ФАЙЛ: outputs_01/news_deduped_2019_2023.csv
Размер (МБ): 22154.2
Используемые колонки: ['Date', 'Stock_symbol', 'Article_title', 'Url', 'Publisher', 'Author', 'Lsa_summary', 'Luhn_summary', 'Textrank_summary', 'Lexrank_summary']


audit nasdaq: 78it [01:47,  1.38s/it]

Всего строк: 15,537,384
Строк после фильтров: 1,232,040 (7.93%)

Сводка по пропускам:


,column,nan_total,nan_total_pct,nan_after_filters,nan_after_filters_pct
5,Author,14351187,92.37,1232040,100.00
4,Publisher,11510741,74.08,901733,73.19
6,Lsa_summary,13045607,83.96,330308,26.81
7,Luhn_summary,13045606,83.96,330307,26.81
8,Textrank_summary,13045606,83.96,330307,26.81
9,Lexrank_summary,13045606,83.96,330307,26.81
0,Date,0,0.00,0,0.00
1,Stock_symbol,9792712,63.03,0,0.00
2,Article_title,1,0.00,0,0.00
3,Url,686,0.00,0,0.00


## Блок 6. Доходности по ценам

Строим доходности по ценовым рядам и сохраняем базовую таблицу

In [10]:
BASE_RET_COLS = ["ticker", "date", "price", "ret_log", "volume"]


def build_prices_returns(prices_dir: str, out_path: str) -> pd.DataFrame:
    rows = []

    for t in tqdm(sorted(TICKER_SET), desc="Reading prices"):
        fp = os.path.join(prices_dir, f"{t}.csv")
        if not os.path.exists(fp):
            continue

        df = pd.read_csv(fp, usecols=["date", "Adj Close", "Volume"], low_memory=False)
        df["date"] = pd.to_datetime(df["date"], errors="coerce").dt.normalize()
        df = df.dropna(subset=["date"])
        df = df[(df["date"] >= START_DATE) & (df["date"] <= END_DATE)]
        df = df.drop_duplicates(subset=["date"]).sort_values("date")

        if df.empty:
            continue

        df["price"] = pd.to_numeric(df["Adj Close"], errors="coerce")
        df["volume"] = pd.to_numeric(df["Volume"], errors="coerce")
        df = df[df["price"] > 0].dropna(subset=["price"])

        if df.empty:
            continue

        df["ret_log"] = np.log(df["price"]).diff()
        df["ticker"] = str(t)

        rows.append(df[BASE_RET_COLS])

    ret = (
        pd.concat(rows, ignore_index=True)
        .sort_values(["ticker", "date"])
        .reset_index(drop=True)
    )
    ret.to_parquet(out_path, index=False)
    print("Сохранено:", out_path)
    return ret


RET_PATH = os.path.join(OUTPUT_DIR, "prices_returns_2019_2023.parquet")

print_step("Построение доходностей по ценам")
if os.path.exists(RET_PATH):
    ret = pd.read_parquet(RET_PATH)
else:
    ret = build_prices_returns(PRICES_DIR, RET_PATH)

ret["ticker"] = ret["ticker"].astype(str).str.strip()
ret["date"] = pd.to_datetime(ret["date"]).dt.normalize()
ret = ret[ret["ticker"].isin(TICKER_SET)].copy()

print("Строк:", f"{len(ret):,}", "Тикеров:", int(ret["ticker"].nunique()))
print("Диапазон дат:", ret["date"].min().date(), "—", ret["date"].max().date())
print("Доля NaN в ret_log, %:", round(float(ret["ret_log"].isna().mean() * 100), 4))


Построение доходностей по ценам
Строк: 2,580,236 Тикеров: 2054
Диапазон дат: 2019-01-02 — 2023-12-28
Доля NaN в ret_log, %: 0.0796


## Блок 7. Рыночная и избыточная доходность

Добавляем рыночную доходность и считаем избыточную доходность относительно SPY

In [12]:
MARKET_TICKER = "SPY"


def build_market_returns(prices_dir: str, ticker: str) -> pd.DataFrame:
    fp = os.path.join(prices_dir, f"{ticker}.csv")
    df = pd.read_csv(fp, usecols=["date", "adj close"], low_memory=False)
    df["date"] = pd.to_datetime(df["date"], errors="coerce").dt.normalize()
    df = df.dropna(subset=["date"]).drop_duplicates(subset=["date"]).sort_values("date")
    df = df[(df["date"] >= START_DATE) & (df["date"] <= END_DATE)]

    df["mkt_price"] = pd.to_numeric(df["adj close"], errors="coerce")
    df = df.dropna(subset=["mkt_price"])
    df["mkt_ret_log"] = np.log(df["mkt_price"]).diff()

    return df[["date", "mkt_ret_log"]]


print_step("Построение рыночной и избыточной доходности")

mkt = build_market_returns(PRICES_DIR, MARKET_TICKER)
print("Рыночный прокси:", MARKET_TICKER)

ret = ret.drop(columns=["mkt_ret_log", "excess_ret_log"], errors="ignore")
ret["mkt_ret_log"] = ret["date"].map(mkt.set_index("date")["mkt_ret_log"])
ret["excess_ret_log"] = ret["ret_log"] - ret["mkt_ret_log"]

print("mkt_ret_log заполнено, %:", round(float(ret["mkt_ret_log"].notna().mean() * 100), 2))
print("excess_ret_log заполнено, %:", round(float(ret["excess_ret_log"].notna().mean() * 100), 2))


Построение рыночной и избыточной доходности
Рыночный прокси: SPY
mkt_ret_log заполнено, %: 99.92
excess_ret_log заполнено, %: 99.92


## Блок 8. Целевые переменные

Строим целевые переменные на горизонтах 1, 2, 3 и 5 торговых дней

In [13]:
print_step("Построение целевых переменных y1, y2, y3 и y5")

ret = ret.sort_values(["ticker", "date"], kind="mergesort").reset_index(drop=True)
g_ex = ret.groupby("ticker", sort=False)["excess_ret_log"]

s1 = g_ex.shift(-1)
s2 = g_ex.shift(-2)
s3 = g_ex.shift(-3)
s4 = g_ex.shift(-4)
s5 = g_ex.shift(-5)

ret["y1_ex"] = s1.astype("float32")
ret["y2_ex"] = (s1 + s2).astype("float32")
ret["y3_ex"] = (s1 + s2 + s3).astype("float32")
ret["y5_ex"] = (s1 + s2 + s3 + s4 + s5).astype("float32")

for c in ["y1_ex", "y2_ex", "y3_ex", "y5_ex"]:
    print(f"{c}: доля NaN = {ret[c].isna().mean()*100:.2f}%")

trade = ret[["ticker", "date"]].rename(columns={"date": "trade_date"}).copy()
trade = trade.sort_values(["ticker", "trade_date"], kind="mergesort").reset_index(drop=True)


Построение целевых переменных y1, y2, y3 и y5
y1_ex: доля NaN = 0.08%
y2_ex: доля NaN = 0.16%
y3_ex: доля NaN = 0.24%
y5_ex: доля NaN = 0.40%


## Блок 9. Сохранение промежуточных таблиц

Сохраняем промежуточные таблицы для модельного ноутбука `02-2_model_processing.ipynb`

In [14]:
print_step("Сохранение промежуточных таблиц для модельного этапа")

RETURNS_TARGETS_PATH = os.path.join(OUTPUT_DIR, "returns_targets_2019_2023.parquet")
TRADE_CALENDAR_PATH = os.path.join(OUTPUT_DIR, "trade_calendar_2019_2023.parquet")

ret.to_parquet(RETURNS_TARGETS_PATH, index=False)
trade.to_parquet(TRADE_CALENDAR_PATH, index=False)

print("Сохранено:", RETURNS_TARGETS_PATH)
print("Сохранено:", TRADE_CALENDAR_PATH)
print("Размер таблицы доходностей и целей:", ret.shape)
print("Размер торгового календаря:", trade.shape)


Сохранение промежуточных таблиц для модельного этапа
Сохранено: outputs_01/returns_targets_2019_2023.parquet
Сохранено: outputs_01/trade_calendar_2019_2023.parquet
Размер таблицы доходностей и целей: (2580236, 12)
Размер торгового календаря: (2580236, 2)
